In [ ]:
%run PIKE.ipynb
%run Ind.ipynb

In [ ]:

import numpy as np
import pandas as pd
import multiprocessing as mp
from pathos.multiprocessing import ProcessingPool as Pool
import time
import os
import traceback



def com_mHSIC(data_X,data_Y,d1,d2,kernelx,kernely):
    p=data_X.shape[1];q=data_Y.shape[1]
    n=data_X.shape[0]
    
    distance_Xlist=com_distance_list(data_X,d1,kernelx)
  
    distance_Ylist=com_distance_list(data_Y,d2,kernely)
    
    alpha_beta_m=com_alpha_beta(distance_Xlist,distance_Ylist,n,p,q)

    atau=np.mean(alpha_beta_m/n/n)

    datay_copy=np.copy(data_Y)
   
    p1_value=[]
    for k in range(100):
        np.random.shuffle(datay_copy)
        distance_Ylist1=com_distance_list(datay_copy,d2,kernely)
        alpha_beta_m1=com_alpha_beta(distance_Xlist,distance_Ylist1,n,p,q)
    
        p1_value.append(np.mean(alpha_beta_m1/n/n))
        
    
    return atau,p1_value



significance_level=0.05
r=0.7
kernelx='np'; kernely='np'
la_candidates = 2**np.linspace(0,-10,15)

def examplepic1(n,p,d):
    data_X=np.reshape(np.random.standard_normal(n*p),(n,p))
    data_Y=np.reshape(np.random.standard_normal(n*p),(n,p))

    
    cov1=np.diag([1.0]*8*2)
    for i in range(8):
        cov1[i,8+i]=0.8
        cov1[8+i,i]=0.8
    mean1=np.zeros(8*2)
    data=np.random.multivariate_normal(mean1, cov1,n)

    data_X[:,:8]=data[:,:8]
    data_Y[:,:8]=data[:,8:]
    return data_X,data_Y
    

def process_single_iteration(args):

    i, n, p, d = args
    
    
    # Set a different random seed for each task
    seed = i
    np.random.seed(seed)
        
    # Generate data
    data_X, data_Y = examplepic1(n, p, d)
        
    # Convert to R matrices
    a = robjects.FloatVector(np.reshape(data_X, (data_X.shape[0] * data_X.shape[1])))
    rdata_X = robjects.r.matrix(a, nrow=n, byrow='TRUE')
        
    b = robjects.FloatVector(np.reshape(data_Y, (data_Y.shape[0] * data_Y.shape[1])))
    rdata_Y = robjects.r.matrix(b, nrow=n, byrow='TRUE')
        
    # Call R functions (ensure these functions are defined in the R environment)
    atau1,atau2,atau3 = com_PIKE(data_X, data_Y, kernelx, kernely, significance_level,r,la_candidates)
    m1,m1list=com_mHSIC(data_X,data_Y,1,1,'np','np')
    m2,m2list=com_mHSIC(data_X,data_Y,1,2,'np','np')
    atau4 = com_significance(m1, m1list, significance_level*100)
    atau5 = com_significance(m2, m2list, significance_level*100)
    atau6 = CGR(rdata_X, rdata_Y, significance_level, 1, "Rademacher")[1]
        
    atau7,atau_bt7=com_HSIC(data_X,data_Y,n,1,1)  # statistic
    atau8,atau_bt8=com_HSIC(data_X,data_Y,n,2,1)
    atau9=test_DC(data_X, data_Y)    # p-value
    atau10=com_PCOV(data_X,data_Y,n)  # p-value
    atau8=mrdcov(rdata_X,rdata_Y)    # p-value
    atau9=Hallin(rdata_X,rdata_Y)    # p-value
    atau10=BCov(rdata_X,rdata_Y)     # p-value

        
    # Return results
    return [\
        atau3,
        atau4,
        atau5,
        np.array(atau6)[0],
        com_significance(atau7, atau_bt7, significance_level*100),
        com_significance(atau8, atau_bt8, significance_level*100),
        np.sum(np.array(atau9) < significance_level),
        np.sum(np.array(atau10) < significance_level)
        ]


        
        
        
def process_batch_parallel(batch_indices, n, p, d, batch_num, total_batches, pool):
    """Process a batch of tasks using process pool"""
    print(f"\nStarting batch {batch_num+1}/{total_batches}, containing {len(batch_indices)} tasks")
    batch_start_time = time.time()
    
    # Prepare arguments
    args_list = [(i, n, p, d) for i in batch_indices]
    
    # Parallel execution (using pool.map)
    batch_results = pool.map(process_single_iteration, args_list)
    
    batch_time = time.time() - batch_start_time
    print(f"Batch {batch_num+1}/{total_batches} completed, time: {batch_time:.2f} seconds")
    print(np.mean(batch_results,0))
    return batch_results



# ========== Process initialization function ==========
def worker_init():
    """Run when each process initializes"""
    import os

    # Set Numba environment
    os.environ['NUMBA_THREADING_LAYER'] = 'workqueue'

    print(f"Process {os.getpid()} initialized")


if __name__ == "__main__":
    for p in [100,400,1600]:
        n = 100
        for d in [0]:
            total_iterations = 300

            def examplepic1(n,p,d):
                data_X=np.reshape(np.random.standard_normal(n*p),(n,p))
                data_Y=np.reshape(np.random.standard_normal(n*p),(n,p))

                
                cov1=np.diag([1.0]*8*2)
                for i in range(8):
                    cov1[i,8+i]=0.8
                    cov1[8+i,i]=0.8
                mean1=np.zeros(8*2)
                data=np.random.multivariate_normal(mean1, cov1,n)

                data_X[:,:8]=data[:,:8]
                data_Y[:,:8]=data[:,8:]
                return data_X,data_Y

    
            # Set number of processes (adjust based on CPU cores)
            n_jobs = min(mp.cpu_count(), 8)  # Use at most 8 processes to avoid R memory pressure
            batch_size = n_jobs * 2  # Each batch processes 2x the number of processes tasks
    
            # Create batch indices
            indices = list(range(total_iterations))
            batches = [indices[i:i + batch_size] for i in range(0, total_iterations, batch_size)]
            total_batches = len(batches)
    
            print(f"Total tasks: {total_iterations}")
            print(f"Number of processes used: {n_jobs}")
            print(f"Batch size: {batch_size}, total batches: {total_batches}")

            # Initialize results dict (adjust keys based on your original code)
            results_dict = {
                'r11': [], 'r21': [], 'r31': [], 'r41': [], 'r51': [],'r61':[],\
                'r71': [], 'r81': []
                # Add other keys as needed
            }

            try:
                # Try to use forkserver (faster than spawn and more friendly to Numba)
                mp.set_start_method('forkserver', force=True)
                print("Using forkserver start method")
            except RuntimeError:
                # If forkserver not available, use spawn
                mp.set_start_method('spawn', force=True)
                print("Using spawn start method")
    
            all_results = []
            start_time = time.time()
    
            # Create process pool (with initialization function)
            with Pool(processes=n_jobs) as pool:
                for batch_num, batch_indices in enumerate(batches):
                    try:
                        batch_results = process_batch_parallel(
                            batch_indices, n, p, d, batch_num, total_batches, pool
                        )
                        all_results.extend(batch_results)
                
                        # Output progress
                        completed = len(all_results)
                        elapsed = time.time() - start_time
                        print(f"Completed {completed}/{total_iterations} tasks, total time: {elapsed:.2f} seconds")
                
                           
                    except Exception as e:
                        print(f"Batch {batch_num+1} processing failed: {str(e)}")
                        continue
    
            # Fill results into dict
            for result in all_results:
                for idx, key in enumerate(results_dict.keys()):
                    if idx < len(result):
                        results_dict[key].append(result[idx])
    
            # Compute averages
            eg1_5 = [results_dict[key] for key in results_dict.keys()]
    
            total_time = time.time() - start_time
            print(f"\nAll tasks completed! Total time: {total_time:.2f} seconds")
            print(f"Average time per task: {total_time/total_iterations:.2f} seconds")
            print(f"Final results: {eg1_5}")
            
            # Save final results
            power = pd.DataFrame(np.mean(np.array([eg1_5[i] for i in range(8)]).T,0))
            filename = f'/n={n}_p={p}_d={d}_pic1.csv'
            power.to_csv(filename)
            print(f"Results saved to: {filename}")
            
            
            


